# 05 - Neuron And MLP

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 课前补丁：只学够用的线性代数

进入 `Neuron` 前，只需要先把 4 个词对上：

| 词 | 在本节里的意思 |
|---|---|
| 标量 | 一个数，比如 `b` |
| 向量 | 一排数，比如 `x = [x1, x2, x3]` 或 `w = [w1, w2, w3]` |
| 点积 | 对应位置相乘，再全部加起来 |
| bias | 最后额外加上的一个可学习标量 `b` |

一个 3 输入神经元可以写成：

$$
act = w_1x_1 + w_2x_2 + w_3x_3 + b
$$

也可以写成更短的线性代数形式：

$$
act = w \cdot x + b
$$

其中：

$$
w \cdot x = w_1x_1 + w_2x_2 + w_3x_3
$$

所以后面看到这行代码时：

```python
act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
```

可以直接翻译成：

```text
把每个 wi 和对应 xi 相乘，再从 bias b 开始累加。
```

`Layer` 也不神秘：一层就是多个神经元。每个神经元都有自己的一条 `w` 向量和一个 `b`，各自对同一个输入向量 `x` 做一次点积。矩阵乘法暂时可以先理解成：**很多个点积排在一起算**。

## 容易混淆：数学里的 `[]` 和 Python 里的 `[]`

这两个东西长得像，但意思不一样：

```text
数学里的 []：排版符号，用来画向量或矩阵的形状
Python 里的 []：list 容器，用来装一组对象
```

先看向量：

```text
数学里常写成列向量：

[ x1 ]
[ x2 ]
[ x3 ]

Python 里常写成 list：

x = [x1, x2, x3]
```

这里不要纠结 Python 里的 list 是横着还是竖着。`x = [x1, x2, x3]` 只是“装了 3 个输入值”。它是不是列向量，要看我们后面怎么用它。

再看矩阵：

```text
数学里的矩阵 W 有 2 行 2 列。
Python 里常用 list of lists 表示：

W = [
    [w11, w12],   # 第 1 行
    [w21, w22],   # 第 2 行
]
```

在神经网络里，可以把矩阵的一行理解成一个神经元的权重。

数学写法用真正的矩阵排版更清楚：

$$
\begin{bmatrix}
w_{11} & w_{12} \\
w_{21} & w_{22}
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
+
\begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}
=
\begin{bmatrix}
y_1 \\
y_2
\end{bmatrix}
$$

把它拆成两行，就是两个神经元分别算一次点积：

$$
y_1 = w_{11}x_1 + w_{12}x_2 + b_1
$$

$$
y_2 = w_{21}x_1 + w_{22}x_2 + b_2
$$

对应到 Python / micrograd 的使用层：

```python
x = [x1, x2]

neuron1.w = [w11, w12]
neuron1.b = b1
y1 = w11*x1 + w12*x2 + b1

neuron2.w = [w21, w22]
neuron2.b = b2
y2 = w21*x1 + w22*x2 + b2

layer(x) = [y1, y2]
```

所以本节先记这条分界线：

```text
数学矩阵：重点看“几行几列”的形状。
Python list：重点看“里面装了哪些对象”。
micrograd 这里没有专门的矩阵类，Neuron / Layer / MLP 都是用 Python list 把很多 Value 组织起来。
```

<details>
<summary><strong>Show / Hide 提示</strong></summary>

点积就是对应位置相乘再相加：

```text
2*4 + (-1)*5 + 3*(-2)
```

`Layer(3, 2)` 表示这一层有 2 个神经元，每个神经元接收 3 个输入，所以每个神经元有 `3 个 w + 1 个 b`。

</details>

## 1. 先手写一个神经元

公式：

$$
n = ReLU(w_1x_1 + w_2x_2 + b)
$$

先用固定小数字，不用随机数。

In [ ]:
x1 = Value(2.0)
x2 = Value(-3.0)
w1 = Value(-1.0)
w2 = Value(0.5)
b = Value(4.0)

act = w1 * x1 + w2 * x2 + b
n = act.relu()

print('act.data =', act.data)
print('n.data =', n.data)

n.backward()
print('w1.grad =', w1.grad)
print('w2.grad =', w2.grad)
print('b.grad  =', b.grad)

观察：如果 `act.data < 0`，ReLU 输出 0，这个神经元这次不会把梯度传回去。现在把 `b` 改成 `4.0` 再运行上一格，看看梯度是否出现。

## 2. 看 `Neuron.__call__`

`Neuron` 的核心就在 `__call__`：对象像函数一样被调用。

In [ ]:
print(inspect.getsource(Neuron.__call__))

把源码翻译成人话：

```text
for 每个输入 xi 和对应权重 wi：
    计算 wi * xi
把这些乘积加起来
再加 b
如果 nonlin=True，就过 ReLU
```

也就是：

$$
out = ReLU(w_1x_1 + w_2x_2 + ... + b)
$$

In [ ]:
random.seed(42)

my_n = Neuron(2)
print('my_n', my_n.parameters())

my_x = [Value(2.0), Value(-3.0)]
my_out = my_n(my_x)
my_out.backward()

print('my_out', my_out)


print(0.2788535969157675 * 2.0 + -0.9499784895546661 * -3.0)

# relu(x) = x when x > 0
# relu(x) = 0 when x <= 0

for i, p in enumerate(my_n.parameters()):
    print(f'param {i}: data={p.data:.4f}, grad={p.grad:.4f}')

## 3. 先把 `Neuron(2)` 画成图

这段代码：

```python
random.seed(42)
n = Neuron(2)

x = [Value(2.0), Value(-3.0)]
out = n(x)
out.backward()
```

先不要把它想成“神经网络”。它就是一个接收 2 个输入的小公式：

```text
输入 x：      [x0, x1]
权重 n.w：   [w0, w1]
bias n.b：   b

act = w0*x0 + w1*x1 + b
out = ReLU(act)   # 如果 nonlin=True
```

画成一张图：

```text
x0 =  2.0 ----* w0 ----\
                         + ---- + b ---- act ---- ReLU ---- out
x1 = -3.0 ----* w1 ----/
```

`Neuron(2)` 的 `2` 只表示：**这个神经元要看 2 个输入，所以它需要 2 个权重**。它仍然只产生 1 个输出。

泛化一下：

```text
Neuron(3) 看 3 个输入 -> 3 个权重 + 1 个 bias -> 1 个输出
Neuron(n) 看 n 个输入 -> n 个权重 + 1 个 bias -> 1 个输出
```

所以 `Neuron(2)` 的参数一定是：

```text
[w0, w1, b]
```

也就是：

```text
2 个输入权重 + 1 个 bias = 3 个参数
```

### 这段代码逐行是什么意思

```python
random.seed(42)
```

固定随机数。这样每次创建 `Neuron(2)`，权重都一样，方便学习和复现。

```python
n = Neuron(2)
```

创建一个神经元。`2` 表示它接收两个输入，所以内部会有：

```text
n.w = [w0, w1]
n.b = b
```

```python
print(n)
```

打印神经元长什么样，比如 `ReLUNeuron(2)`，意思是：带 ReLU 的 2 输入神经元。

```python
print('parameters:', n.parameters())
```

打印这个神经元的可训练参数：

```text
[w0, w1, b]
```

```python
x = [Value(2.0), Value(-3.0)]
```

准备两个输入。这里的输入也用 `Value` 包起来，是为了让它们能参与 micrograd 的计算图。

```python
out = n(x)
```

因为 `Neuron` 定义了 `__call__`，所以 `n(x)` 会执行神经元的前向计算：

```text
out = ReLU(w0*x0 + w1*x1 + b)
```

```python
out.backward()
```

从输出 `out` 往回算梯度。之后每个参数的 `.grad` 就表示：

```text
out 对这个参数的偏导
```

```python
for i, p in enumerate(n.parameters()):
    print(f'param {i}: data={p.data:.4f}, grad={p.grad:.4f}')
```

逐个打印参数的当前值 `data` 和它收到的梯度 `grad`。

### backward 后的梯度图

这次前向图是：

```text
x0 =  2.0 ----* w0 ----\
                         + ---- + b ---- act ---- ReLU ---- out
x1 = -3.0 ----* w1 ----/
```

反向传播后，参数梯度可以这样看：

```text
out.grad = 1

w0.grad = x0 * out.grad =  2.0
w1.grad = x1 * out.grad = -3.0
b.grad  = 1  * out.grad =  1.0
```

所以打印结果里：

```text
param 0: data= 0.2789, grad= 2.0000   # w0
param 1: data=-0.9500, grad=-3.0000   # w1
param 2: data= 0.0000, grad= 1.0000   # b
```

这里的 `grad` 不是“参数自己的变化”，而是：

```text
如果这个参数稍微变大一点，out 会怎么变。
```

## 4. 用真正的 Neuron 跑一次

In [ ]:
random.seed(42)
n = Neuron(2)
print(n)
print('parameters:', n.parameters())

x = [Value(2.0), Value(-3.0)]
out = n(x)
print('out =', out)

out.backward()
for i, p in enumerate(n.parameters()):
    print(f'param {i}: data={p.data:.4f}, grad={p.grad:.4f}')

## 5. 把 `MLP(2, [3, 1])` 画开

先不要把它当成神秘模型。`MLP(nin, nouts)` 只有两个信息：

```text
nin   = 原始输入有几个数
nouts = 每一层要输出几个数
```

所以：

```python
model = MLP(2, [3, 1])
```

先代入变量：

```text
nin   = 2
nouts = [3, 1]
```

micrograd 源码里第一步会做：

```python
sz = [nin] + nouts
```

所以这里就是：

```text
sz = [2] + [3, 1]
sz = [2, 3, 1]
```

关键来了：`sz` 里面相邻两个数字，配成一层。

| i | 取哪两个数 | 变成哪一层 | 这层是什么意思 |
|---|---|---|---|
| 0 | `sz[0] -> sz[1]`，也就是 `2 -> 3` | `Layer(2, 3)` | 接收 2 个输入，造 3 个神经元，所以输出 3 个数 |
| 1 | `sz[1] -> sz[2]`，也就是 `3 -> 1` | `Layer(3, 1)` | 接收上一层的 3 个输出，造 1 个神经元，所以输出 1 个数 |

所以这句：

```text
2 个输入 -> 第一层 3 个神经元 -> 第二层 1 个神经元 -> 1 个输出
```

不是直接凭空说的，而是从 `sz = [2, 3, 1]` 这样算出来的：

```text
[2, 3, 1]
 │  │
 │  └─ 第一层输出 3 个数，所以第一层有 3 个神经元
 └──── 原始输入有 2 个数

[2, 3, 1]
    │  │
    │  └─ 第二层输出 1 个数，所以第二层有 1 个神经元
    └──── 第二层接收上一层的 3 个输出
```

画成结构图：

```text
输入 x: [x0, x1]
        │
        ▼
Layer(2, 3)
  ├─ Neuron(2) -> h0
  ├─ Neuron(2) -> h1
  └─ Neuron(2) -> h2
        │
        ▼
隐藏层输出 h: [h0, h1, h2]
        │
        ▼
Layer(3, 1)
  └─ Neuron(3) -> y
        │
        ▼
最终输出: y
```

再拆得更细一点：

```text
Layer(2, 3)
= 3 个 Neuron(2)
= 3 个神经元都看同一个输入 [x0, x1]
= 每个神经元都有自己的 [w0, w1, b]
= 分别算出 h0、h1、h2
= 输出 [h0, h1, h2]
```

这里很容易少算 bias：

```text
不是：3 个神经元共享 1 个 bias
而是：3 个神经元各有 1 个 bias
```

然后：

```text
Layer(3, 1)
= 1 个 Neuron(3)
= 这个神经元看上一层输出 [h0, h1, h2]
= 算出最终 y
```

注意两个容易混的点：

```text
2      不是“2 层”，而是原始输入有 2 个数。
[3, 1] 不是输入，而是“每一层的输出宽度清单”。
```

这里第一层默认带 ReLU，最后一层不带 ReLU，所以 `print(model)` 里会看到：

```text
第一层：ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2)
第二层：LinearNeuron(3)
```

为什么最后一层不用 ReLU？因为最后输出通常要保留原始预测值，后面再交给 loss 去判断它大了还是小了。




In [ ]:
random.seed(42)

# 先手动模拟 MLP 构造函数里的结构计算。
nin = 2
nouts = [3, 1]
sz = [nin] + nouts
print('sz =', sz)

for i in range(len(nouts)):
    layer_in = sz[i]
    layer_out = sz[i + 1]
    print(f'第 {i + 1} 层: Layer({layer_in}, {layer_out})')

# 所以 MLP(2, [3, 1]) 的意思是：
# sz = [2, 3, 1]
# 2 -> 3: 第一层 Layer(2, 3)，3 个 Neuron(2)，输出 [h0, h1, h2]
# 3 -> 1: 第二层 Layer(3, 1)，1 个 Neuron(3)，输出 y
model = MLP(2, [3, 1])

print(model)
print('parameters:', model.parameters())
print('number of parameters:', len(model.parameters()))

# 输入必须有 2 个 Value，因为 MLP 的 nin=2。
x = [Value(1.0), Value(2.0)]

# y 是最终输出；中间的 h0/h1/h2 被 Layer 和 MLP 帮你藏起来了。
y = model(x)
print('prediction:', y)

y.backward()
print('first 5 parameter grads:')
for p in model.parameters()[:5]:
    print(p)



## 6. 过关小测

对于：

```python
MLP(2, [3, 1])
```

先不要直接背结构。先算 `sz`：

```text
nin = 2
nouts = [3, 1]
sz = [nin] + nouts = [2, 3, 1]
```

然后相邻两项配对：

```text
2 -> 3 变成 Layer(2, 3)
3 -> 1 变成 Layer(3, 1)
```

所以结构是：

```text
输入层宽度：2
隐藏层宽度：3
输出层宽度：1
```

再说它会创建哪些对象：

```text
第 1 层：Layer(2, 3)
        = 3 个 Neuron(2)
        = 每个神经元接收 [x0, x1]，各自产生一个 h
        = 输出 [h0, h1, h2]

第 2 层：Layer(3, 1)
        = 1 个 Neuron(3)
        = 接收 [h0, h1, h2]，产生最终 y
```

最后数参数。关键是：**每个神经元都有自己的 bias，不是整层共用一个 bias**。

```text
一个 Neuron(2) = 2 个 w + 1 个 b = 3 个参数
Layer(2, 3)   = 3 个 Neuron(2) = 3 * 3 = 9 个参数
```

第一层展开就是：

```text
第 1 个 Neuron(2): w00, w01, b0  -> 3 个参数
第 2 个 Neuron(2): w10, w11, b1  -> 3 个参数
第 3 个 Neuron(2): w20, w21, b2  -> 3 个参数

第一层总数 = 3 + 3 + 3 = 9
```

第二层：

```text
一个 Neuron(3) = 3 个 w + 1 个 b = 4 个参数
Layer(3, 1)   = 1 个 Neuron(3) = 4 个参数
```

所以总数：

```text
第一层 9 + 第二层 4 = 13
```

你看到的 `parameters` 列表也正好是这个顺序：

```text
前 9 个：第一层 3 个 ReLUNeuron(2)，每个 [w, w, b]
后 4 个：第二层 1 个 LinearNeuron(3)，也就是 [w, w, w, b]
```



In [ ]:
model = MLP(2, [3, 1])
assert len(model.parameters()) == 13
print('OK: parameter count =', len(model.parameters()))

## What To Remember

这一节记住六句话：

```text
1. 数学里的 [] 是向量/矩阵的排版，Python 里的 [] 是 list 容器。
2. 点积就是对应位置相乘再相加：w1*x1 + w2*x2 + ...
3. 神经元不是新魔法，只是很多个 Value 运算拼出来。
4. 参数就是 w 和 b，它们也是 Value。
5. forward 后得到预测值，backward 后每个参数都有 grad。
6. MLP = Layer 的列表，Layer = Neuron 的列表。
```

下一节才开始训练，也就是用 `p.data += -learning_rate * p.grad` 更新参数。

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - Neuron(3)

In [ ]:
student_neuron3 = None  # [input_count, output_count, parameter_count]



qa_check('qa_check_class_05_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 把形状说成人话：`Neuron(n)` 接收 n 个输入、输出 1 个数、参数是 n 个权重加 1 个 bias。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_neuron3` 填 `[3, 1, 4]`。

</details>

## Run - 手算一个神经元

In [ ]:
w = [0.2, -0.5]
x = [2.0, -3.0]
b = 0.1
act = sum((wi*xi for wi, xi in zip(w, x)), b)
print('act =', act)
print('formula: w0*x0 + w1*x1 + b')

## 拆开看 - Layer 和 MLP

In [ ]:
from micrograd.nn import Layer, MLP
random.seed(42)
print('Layer(2,3):', Layer(2,3))
print('MLP(2,[3,1]):', MLP(2,[3,1]))

## Modify - 参数数量

In [ ]:
# MLP(2,[3,1]) 参数数量
student_param_count = None



qa_check('qa_check_class_05_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 把形状说成人话：`Neuron(n)` 接收 n 个输入、输出 1 个数、参数是 n 个权重加 1 个 bias。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_param_count` 填 `13`。

</details>